# Main file

## imports

In [ ]:
import torch
import wandb
import numpy as np

In [ ]:
import random

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# optional: more deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False    

In [17]:
from model.clip import model as clip_model, processor as clip_processor
from fairface_vit import FairFaceViT

In [18]:
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import get_train_transform, get_age_transform, get_val_transform

In [19]:
from training.train import train_loop, test_loop

In [20]:
from training.losses import get_age_weights, get_loss_function

## Hyperparameters and WandB initialization

In [21]:
# wandb.login()

In [22]:
# print(wandb.__version__)
# print(wandb.login())

In [ ]:
import os
model_name = 'clip' 

os.makedirs(f"../checkpoints/{model_name}", exist_ok=True)

epochs         = 20
batch_size     = 16

learning_rate  = 1e-4
weight_decay   = 1e-2
age_dropout1   = 0.15
age_hidden_dim = 384 
loss_weights  = {
    "gender":1,
    "age":1,
    "race":1
}

wandb.init(
    project="fairface-vit",
    name="clip-vit-base-patch16",
    id="clip-vit-base-patch16", 

    resume="allow",
    mode="offline",
    
    config={
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "age_dropout1": age_dropout1,
        "age_hidden_dim": age_hidden_dim,
        "weight_decay": weight_decay,
        "optimizer": "AdamW",
        "model": "clip-vit-base-patch16",
        "loss_weights": loss_weights
    }
)

wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id clip-vit-base-patch16.


In [24]:
print(wandb.run)

## Model and Device

In [25]:
fairface_clip_model = FairFaceViT(clip_model, age_dropout1, age_hidden_dim)

start freezing the backbone parameters...

finished freezing the backbone parameters...



In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fairface_clip_model.to(device)

FairFaceViT(
  (backbone): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)

## Dataset and Dataloader

In [27]:
train_set, val_set = get_dataset(
    get_train_transform(clip_processor.image_processor),
    get_age_transform(clip_processor.image_processor),
    get_val_transform(clip_processor.image_processor)
)


train_dataloader, val_dataloader = get_dataloaders(
    train_set,
    val_set,
    batch_size=batch_size,
    num_workers=0,
)

## Loss functions


In [28]:
age_labels = np.array(train_set.dataset["age"])

age_weights = get_age_weights(
    age_labels,
    num_classes=9,
    device=device
)

loss_funcs = get_loss_function(age_weights)

## Optimizer

In [29]:

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fairface_clip_model.parameters()),
    lr=learning_rate,
    weight_decay=weight_decay
)


## Traning

In [ ]:
best_acc = 0
patience = 5
patience_counter = 0
min_delta = 0.002

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_task_loss, train_metrics = train_loop(
        train_dataloader, fairface_clip_model, loss_funcs, 
        loss_weights, optimizer, device, epoch+1, epochs
    )

    val_loss, val_task_loss, metrics, subgroup_metrics = test_loop(
        val_dataloader,fairface_clip_model, loss_funcs, 
        loss_weights, device, epoch+1, epochs
    )

    log_dict = {

        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        
        # losses
        "train/loss": train_loss,
        "val/loss": val_loss,


        "train/gender_loss": train_task_loss["gender"],
        "train/age_loss": train_task_loss["age"],
        "train/race_loss": train_task_loss["race"],


        "val/gender_loss": val_task_loss["gender"],
        "val/age_loss": val_task_loss["age"],
        "val/race_loss": val_task_loss["race"],
    }

    # train metrics
    for task, values in train_metrics.items():
        for metric_name, value in values.items():
            log_dict[f"train/{task}/{metric_name}"] = value

    # val metrics
    for task, values in metrics.items():
        for metric_name, value in values.items():
            log_dict[f"val/{task}/{metric_name}"] = value

    # subgroup accuracy
    for group_name, values in subgroup_metrics.items():
        for subgroup, acc in values.items():
            log_dict[f"subgroup/{group_name}/{subgroup}"] = acc

    current_acc = (
        metrics["gender"]["accuracy"]
        + metrics["age"]["accuracy"]
        + metrics["race"]["accuracy"]
    ) / 3

    log_dict["val/avg_accuracy"] = current_acc

    wandb.log(log_dict)

    if current_acc > best_acc + min_delta:
        
        best_acc = current_acc
        patience_counter = 0

        torch.save(
            {
                "gender_head": fairface_clip_model.gender.state_dict(),
                "age_head": fairface_clip_model.age.state_dict(),
                "race_head": fairface_clip_model.race.state_dict(),
            },
            f"../checkpoints/{model_name}/{model_name}-best_heads.pt",
        )

        print(f"Saved {model_name}-best_heads.pt | avg accuracy={best_acc:.4f}")

    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

    torch.save(
        {
            "gender_head": fairface_clip_model.gender.state_dict(),
            "age_head": fairface_clip_model.age.state_dict(),
            "race_head": fairface_clip_model.race.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),

            "epoch": epoch,
            "best_acc": best_acc,

            "patience": patience,
            "patience_counter": patience_counter,
            "min_delta": min_delta,

             # RNG states
            "python_rng_state": random.getstate(),
            "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),

            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
        ),
            
        },
        f"../checkpoints/{model_name}/{model_name}-checkpoint.pt",   
    )

    if patience_counter >= patience:
        print(f"\nEarly stopping after {epoch+1} epochs.")
        break



Epoch 1/20


 avg loss:        2.0031
 avg age loss:    0.97
 avg gender loss: 0.18
 avg race loss:   0.85

 gender: acc=0.939  f1=0.934
 age:    acc=0.562  f1=0.554  mae=0.506
 race:   acc=0.694  f1=0.684


 avg loss: 1.7683
 gender: acc=0.955  f1=0.953
 age:    acc=0.573  f1=0.579   mae=0.474
 race:   acc=0.723  f1=0.718
Saved clip-best_heads.pt | avg accuracy=0.7505

Epoch 2/20


 avg loss:        1.7737
 avg age loss:    0.89
 avg gender loss: 0.13
 avg race loss:   0.75

 gender: acc=0.950  f1=0.946
 age:    acc=0.581  f1=0.578  mae=0.469
 race:   acc=0.721  f1=0.713


 avg loss: 1.7673
 gender: acc=0.956  f1=0.954
 age:    acc=0.576  f1=0.549   mae=0.481
 race:   acc=0.727  f1=0.722
Saved clip-best_heads.pt | avg accuracy=0.7531

Epoch 3/20


...

 avg loss:        1.6971
 avg age loss:    0.85
 avg gender loss: 0.13
 avg race loss:   0.72

 gender: acc=0.951  f1=0.948
 age:    acc=0.596  f1=0.601  mae=0.452
 race:   acc=0.729  f1=0.722


 avg loss: 1.7489
 gender: acc=0.956  f1=0.953
 age:    acc=0.570  f1=0.555   mae=0.488
 race:   acc=0.733  f1=0.727
No improvement (3/5)

Epoch 7/20


 avg loss:        1.6859
 avg age loss:    0.84
 avg gender loss: 0.13
 avg race loss:   0.72

 gender: acc=0.952  f1=0.948
 age:    acc=0.599  f1=0.605  mae=0.449
 race:   acc=0.731  f1=0.725


 avg loss: 1.7332
 gender: acc=0.957  f1=0.954
 age:    acc=0.586  f1=0.583   mae=0.461
 race:   acc=0.734  f1=0.726
No improvement (4/5)

Epoch 8/20


KeyboardInterrupt: 

In [ ]:
wandb.finish()